# oz-tracker — Opportunity Zone Investment Tracker
## Demo: OZ 1.0 / OZ 2.0 screening & QOF tax-benefit modeling

---

## ⚠️ READ FIRST — where this notebook's tract data comes from

**Every tract lookup in this notebook runs on the built-in SYNTHETIC SAMPLE
SET, via the explicit `OZ1Checker.from_sample()` / `OZ2Checker.from_sample()`
entry points. It is 8 fake tracts and 7 fake rows with invented economic
values. It is demo data for exercising the API shape, and it is NEVER a valid
answer to a real Opportunity Zone question.**

### Every GEOID that appears below, classified

Checked against `~/recon-2026-07-30/geo/universes.pkl` using the corrected
detector — a tract is *real now* if it appears in the **2024 Gazetteer ∪ the
2020 relationship file**, not in the 2020 universe alone (which
false-positives all 884 Connecticut tracts).

An earlier version of this banner listed only five non-existent GEOIDs and
said nothing about the rest. That is worse than saying nothing: a partial
disclosure list implies the unlisted ones were checked and passed. This list
is complete for every GEOID printed anywhere in this notebook.

| GEOID | Label used here | Status |
|---|---|---|
| `17031840100` | Chicago South Side | **real, current** |
| `17031839100` | Chicago West Side | **real, current** |
| `17031010100` | Chicago North Shore | **real, current** — and deliberately absent from the sample |
| `36061015900` | NYC Bronx | **real, current** |
| `36061019100` | *(sample set only, not displayed)* | **real, current** |
| `13121010400` | Atlanta | **REAL BUT STALE** — exists only in the 2010 vintage; it is not a current tract |
| `36047052200` | NYC Brooklyn Affluent | **DOES NOT EXIST** in any vintage |
| `26163518300` | Detroit | **DOES NOT EXIST** in any vintage |
| `17019000100` | Rural Illinois | **DOES NOT EXIST** in any vintage |
| `48113010900` | *(sample set only, not displayed)* | **DOES NOT EXIST** in any vintage |
| `26163520100` | *(sample set only, not displayed)* | **DOES NOT EXIST** in any vintage |
| `26001010100` | *(sample set only, not displayed)* | **DOES NOT EXIST** in any vintage |

So six of the eleven GEOIDs in this notebook are invented, and a seventh
(`13121010400`) is a real 2010-vintage tract that no longer exists — presented
below beside genuinely current tracts under equally plausible city labels,
with nothing in the output distinguishing them. That is the point of this
banner: **the labels are decoration, and the sample verdicts are fiction.**

### The real lookup does not work in 0.2.0

Both upstream sources return HTTP 404, so `OZ1Checker()` and `OZ2Checker()`
**raise `OZDownloadError`** rather than answer — section 0 below demonstrates
that directly. See the README "Status" section.

**Tri-state results.** `is_designated()` / `is_eligible()` / `is_rural()`
return `True` or `None` — **never `False`**. `True` is a fact; `None` means
"could not determine", which is *not* a negative finding. Every rendering in
this notebook branches on `is True` / `is None`, never on truthiness — a bare
`if designated:` would collapse `None` into the same branch as a real negative
and print a confident "NO" for a tract nobody checked. That is precisely the
fabricated negative 0.2.0 exists to remove.

The tax-benefit, scenario and portfolio sections (5 onward) perform **no tract
lookup at all** and are fully functional — those numbers are real outputs of
the shipped calculators, computed from the inputs shown.

---

**Key context:** The One Big Beautiful Bill Act (July 2025) made Opportunity
Zones permanent. OZ 2.0 takes effect January 1, 2027 with stricter eligibility,
enhanced rural benefits, and new reporting requirements.

In [1]:
import sys
sys.path.insert(0, '..')

from oztracker import (
    OZ1Checker, OZ2Checker,
    OZInvestment, calculate_benefits, compare_scenarios,
    OZPortfolio, FUND_TYPES, OZ_VERSIONS, OZDownloadError,
)
from oztracker.data.loader import check_oz1_eligibility, check_oz2_eligibility
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("oz-tracker loaded successfully")
print(f"\nOZ Program Versions:")
for k, v in OZ_VERSIONS.items():
    print(f"  {k}: {v}")


def tri(value):
    """Render a tri-state lookup result.

    The whole point: `None` renders as NOT CONFIRMED, never as "NO". Branching
    on `is True` / `is None` rather than truthiness is mandatory — `None` is
    falsy, so `"YES" if value else "NO"` would print a confident negative for
    a tract the package could not answer for.
    """
    if value is True:
        return "YES"
    if value is None:
        return "NOT CONFIRMED"
    return "NO"   # unreachable in 0.2.0; kept so the mapping stays total

oz-tracker loaded successfully

OZ Program Versions:
  oz1: OZ 1.0 — Tax Cuts and Jobs Act 2017 (expires 2028)
  oz2: OZ 2.0 — One Big Beautiful Bill Act 2025 (effective 2027, permanent)


## 0. The real lookup raises — this is the headline 0.2.0 behavior

Before any demo data: what happens when you ask for the *real* designation
list. Through 0.1.0 this silently substituted the 8-row sample set and answered
`False` for 99.91% of genuinely designated tracts. It now refuses.

In [2]:
try:
    OZ1Checker()
    print("UNEXPECTED: constructed against real data")
except OZDownloadError as e:
    print("OZ1Checker() raised OZDownloadError, as designed:\n")
    print(f"  {e}\n")
    print("  -> no fabricated answer, no silent sample substitution")

OZ1Checker() raised OZDownloadError, as designed:

  Failed to download OZ 1.0 tract list from https://www.cdfifund.gov/sites/cdfi/files/2018-06/QOZ_Tracts_List_Formatted_July2018.xlsx: not found (404) — this URL is known-dead as of 2026-07-30 and has not been repointed in 0.2.0; see the oz-tracker README

  -> no fabricated answer, no silent sample substitution


## 1. OZ 1.0 designation — SAMPLE DATA

8,764 census tracts were designated under IRS Notice 2018-48, on **2010**
census-tract boundaries. We cannot reach that list (404), so the check below
runs against the 8-tract synthetic sample.

Watch the `17031010100` row: it is a **real** Cook County tract, and it is
absent from the sample, so the honest answer is NOT CONFIRMED. Rendering that
as "NO" would be a fabricated negative.

In [3]:
oz1 = OZ1Checker.from_sample()
print(f"data_source = {oz1.data_source!r}   <- synthetic, not the designation list")
print(oz1)

tracts = [
    ("17031840100", "Chicago South Side"),
    ("17031839100", "Chicago West Side"),
    ("17031010100", "Chicago North Shore"),
    ("26163518300", "Detroit"),
    ("36061015900", "NYC Bronx"),
    ("36047052200", "NYC Brooklyn Affluent"),
    ("13121010400", "Atlanta"),
    ("17019000100", "Rural Illinois"),
]

print(f"\n{'Tract ID':<15} {'Location':<25} {'OZ 1.0 Designated'}")
print("-" * 62)
for tract_id, location in tracts:
    designated = oz1.is_designated(tract_id)
    print(f"{tract_id:<15} {location:<25} {tri(designated)}")

print(f"\nTotal tracts in this SAMPLE set: {oz1.tract_count:,}")
print("The real OZ 1.0 universe is 8,764 tracts. This is 8.")
print("\nNOT CONFIRMED != not designated. It means the question was not answered.")

data_source = 'sample'   <- synthetic, not the designation list
OZ1Checker(data_source='sample', tracts=8)

Tract ID        Location                  OZ 1.0 Designated
--------------------------------------------------------------
17031840100     Chicago South Side        YES
17031839100     Chicago West Side         YES
17031010100     Chicago North Shore       NOT CONFIRMED
26163518300     Detroit                   YES
36061015900     NYC Bronx                 YES
36047052200     NYC Brooklyn Affluent     NOT CONFIRMED
13121010400     Atlanta                   YES
17019000100     Rural Illinois            NOT CONFIRMED

Total tracts in this SAMPLE set: 8
The real OZ 1.0 universe is 8,764 tracts. This is 8.

NOT CONFIRMED != not designated. It means the question was not answered.


## 2. OZ 2.0 eligibility — SAMPLE DATA

OZ 2.0 uses stricter criteria:
- MFI < **70%** of AMI (vs 80% in OZ 1.0), OR
- Poverty rate >= 20% AND MFI <= 125% of AMI

Contiguous tract rule eliminated — fewer eligible tracts overall.

Again: synthetic frame, invented economic values, tri-state rendering.

In [4]:
oz2 = OZ2Checker.from_sample()
print(f"data_source = {oz2.data_source!r}   <- synthetic, not Rev. Proc. 2026-14")

print(f"\n{'Tract ID':<15} {'Location':<25} {'OZ 2.0 Eligible':<18} {'Rural'}")
print("-" * 75)
for tract_id, location in tracts:
    eligible = oz2.is_eligible(tract_id)
    rural = oz2.is_rural(tract_id)
    print(f"{tract_id:<15} {location:<25} {tri(eligible):<18} {tri(rural)}")

print(f"\nEligible tracts in SAMPLE: {oz2.eligible_tract_count:,}")
print(f"Rural tracts in SAMPLE:    {oz2.rural_tract_count:,}")
print("\nFor reference, the real Rev. Proc. 2026-14 Appendix lists 25,332")
print("eligible tracts, 8,334 of them rural. This frame has 7 rows.")

OZ 2.0: 7 likely eligible tracts
OZ 2.0: 2 rural tracts
data_source = 'sample'   <- synthetic, not Rev. Proc. 2026-14

Tract ID        Location                  OZ 2.0 Eligible    Rural
---------------------------------------------------------------------------
17031840100     Chicago South Side        YES                NOT CONFIRMED
17031839100     Chicago West Side         YES                NOT CONFIRMED
17031010100     Chicago North Shore       NOT CONFIRMED      NOT CONFIRMED
26163518300     Detroit                   YES                NOT CONFIRMED
36061015900     NYC Bronx                 YES                NOT CONFIRMED
36047052200     NYC Brooklyn Affluent     NOT CONFIRMED      NOT CONFIRMED
13121010400     Atlanta                   YES                NOT CONFIRMED
17019000100     Rural Illinois            YES                YES

Eligible tracts in SAMPLE: 7
Rural tracts in SAMPLE:    2

For reference, the real Rev. Proc. 2026-14 Appendix lists 25,332
eligible tracts, 8,334 

## 3. OZ 1.0 vs OZ 2.0 tract comparison — SAMPLE DATA

Set arithmetic over the two sample sets. This is a comparison of what each
*source* contains, not a per-tract verdict, so the tri-state contract does not
apply — but the counts are meaningless as real intelligence, because both
inputs are synthetic.

In [5]:
comparison = oz2.compare_oz1_oz2(oz1.designated_tracts)
print("Counts above are over 8 and 7 SYNTHETIC tracts respectively.")
print("They describe the demo fixtures, not the OZ program.")


OZ 1.0 vs OZ 2.0 Tract Comparison
  OZ 1.0 designated:      8
  OZ 2.0 eligible:        7
  In both programs:       5
  OZ 1.0 only (losing):   3
  OZ 2.0 only (gaining):  2

Counts above are over 8 and 7 SYNTHETIC tracts respectively.
They describe the demo fixtures, not the OZ program.


## 4. Eligibility rule comparison — FULLY FUNCTIONAL, no lookup

`check_oz1_eligibility()` / `check_oz2_eligibility()` are pure threshold
functions over poverty-rate and AMI-ratio values **you** supply. They perform
no lookup and cannot fabricate anything, so they return a genuine `bool` and
YES/NO is the correct rendering here — unlike the tract lookups above.

In [6]:
test_cases = [
    (0.38, 0.55, "High poverty, low AMI"),
    (0.42, 0.48, "Very high poverty, very low AMI"),
    (0.18, 0.92, "Low poverty, high AMI"),
    (0.25, 0.75, "Moderate poverty, moderate AMI"),
    (0.15, 0.68, "Low poverty, below 70% AMI"),
    (0.22, 1.10, "Moderate poverty, above AMI threshold"),
]

print(f"{'Poverty':<10} {'AMI Ratio':<12} {'Description':<35} {'OZ 1.0':<10} {'OZ 2.0'}")
print("-" * 80)
for pr, ami, desc in test_cases:
    oz1_elig = check_oz1_eligibility(pr, ami)
    oz2_elig = check_oz2_eligibility(pr, ami)
    # genuine bools from a pure function - YES/NO is honest here
    print(f"{pr*100:.0f}%{'':<7} {ami*100:.0f}%{'':<9} {desc:<35} "
          f"{'YES' if oz1_elig else 'NO':<10} {'YES' if oz2_elig else 'NO'}")

Poverty    AMI Ratio    Description                         OZ 1.0     OZ 2.0
--------------------------------------------------------------------------------
38%        55%          High poverty, low AMI               YES        YES
42%        48%          Very high poverty, very low AMI     YES        YES
18%        92%          Low poverty, high AMI               NO         NO
25%        75%          Moderate poverty, moderate AMI      YES        YES
15%        68%          Low poverty, below 70% AMI          YES        YES
22%        110%          Moderate poverty, above AMI threshold YES        YES


## 5. QOF Tax Benefit Calculator — OZ 2.0 Standard

A $500k capital gain invested in an OZ 2.0 QOF on March 15, 2027.
Held for 10 years and exited on March 15, 2037.


In [7]:
standard_inv = OZInvestment(
    id="INV001",
    investor_name="Jay Patel",
    fund_name="Midwest OZ Fund I",
    fund_type="qof",
    oz_version="oz2",
    tract_id="17031840100",
    investment_type="real_estate",
    capital_gain_invested=500_000,
    investment_date="2027-03-15",
    fmv_at_investment=500_000,
    current_fmv=750_000,
    state="IL",
    is_rural=False,
)

benefits_std = calculate_benefits(
    standard_inv,
    current_fmv=750_000,
    exit_date="2037-03-15",
)
benefits_std.summary()



QOF Tax Benefit Summary — Midwest OZ Fund I
  Investor:              Jay Patel
  Fund Type:             QOF
  OZ Version:            OZ2
  Rural Benefits:        No
  Capital Gain Invested: $500,000
  Holding Period:        10.0 years

  Deferred Gain:         $500,000
  Step-Up Amount:        $50,000 (10%)
  Deferred Tax Savings:  $17,255
  Excluded Appreciation: $250,000
  ── Total Tax Benefit:  $76,755
  ── Effective Tax Rate: 8.5%



## 6. Rural QORF — Enhanced 30% Step-Up Benefit

The same investment in a rural Qualified Opportunity Rural Fund (QORF)
receives triple the basis step-up: 30% vs 10%.


In [8]:
rural_inv = OZInvestment(
    id="INV002",
    investor_name="Jay Patel",
    fund_name="Rural Illinois QORF",
    fund_type="qorf",
    oz_version="oz2",
    tract_id="17019000100",
    investment_type="real_estate",
    capital_gain_invested=500_000,
    investment_date="2027-03-15",
    fmv_at_investment=500_000,
    current_fmv=750_000,
    state="IL",
    is_rural=True,
)

benefits_rural = calculate_benefits(
    rural_inv,
    current_fmv=750_000,
    exit_date="2037-03-15",
)
benefits_rural.summary()

print(f"Standard step-up:  {benefits_std.stepup_pct*100:.0f}%  (${benefits_std.stepup_amount:,.0f})")
print(f"Rural step-up:     {benefits_rural.stepup_pct*100:.0f}%  (${benefits_rural.stepup_amount:,.0f})")
print(f"Rural advantage:   ${benefits_rural.total_tax_benefit - benefits_std.total_tax_benefit:,.0f} additional benefit")



QOF Tax Benefit Summary — Rural Illinois QORF
  Investor:              Jay Patel
  Fund Type:             QORF
  OZ Version:            OZ2
  Rural Benefits:        Yes
  Capital Gain Invested: $500,000
  Holding Period:        10.0 years

  Deferred Gain:         $500,000
  Step-Up Amount:        $150,000 (30%)
  Deferred Tax Savings:  $39,865
  Excluded Appreciation: $250,000
  ── Total Tax Benefit:  $99,365
  ── Effective Tax Rate: 3.9%

Standard step-up:  10%  ($50,000)
Rural step-up:     30%  ($150,000)
Rural advantage:   $22,610 additional benefit


## 7. Full Scenario Comparison

Compare all four strategies for a $1MM capital gain:
1. No OZ investment — pay tax immediately
2. OZ 1.0 standard QOF
3. OZ 2.0 standard QOF
4. OZ 2.0 Rural QORF (enhanced)


In [9]:
scenarios = compare_scenarios(
    capital_gain=1_000_000,
    investment_date="2027-01-01",
    current_fmv=1_400_000,
    exit_date="2037-01-01",
)

print(f"\nSummary — $1MM Capital Gain, 10-year hold, $1.4MM exit FMV:")
print(f"  No OZ:          Tax = ${scenarios['no_oz_tax']:,.0f}")
print(f"  OZ 1.0:         Benefit = ${scenarios['oz1_benefit']:,.0f}")
print(f"  OZ 2.0:         Benefit = ${scenarios['oz2_benefit']:,.0f}")
print(f"  OZ 2.0 Rural:   Benefit = ${scenarios['oz2_rural_benefit']:,.0f}")



OZ Scenario Comparison — $1,000,000 Capital Gain
  No OZ Investment:     Tax = $238,000 (rate: 23.8%)
  OZ 1.0 Standard:      Tax benefit = $141,015
  OZ 2.0 Standard:      Tax benefit = $129,710
  OZ 2.0 Rural QORF:    Tax benefit = $174,930
  Best strategy:        Rural QORF


Summary — $1MM Capital Gain, 10-year hold, $1.4MM exit FMV:
  No OZ:          Tax = $238,000
  OZ 1.0:         Benefit = $141,015
  OZ 2.0:         Benefit = $129,710
  OZ 2.0 Rural:   Benefit = $174,930


## 8. Holding Period Sensitivity

How do benefits change as the holding period increases?


In [10]:
holding_periods = [3, 5, 7, 10, 15, 20]

print(f"{'Hold (yrs)':<12} {'Step-Up %':<12} {'Exclusion ($)':<18} {'Total Benefit ($)'}")
print("-" * 60)

for years in holding_periods:
    from datetime import datetime
    from dateutil.relativedelta import relativedelta

    exit_dt = (datetime(2027, 1, 1) + relativedelta(years=years)).strftime("%Y-%m-%d")
    b = calculate_benefits(
        OZInvestment(
            id=f"TEST{years}", investor_name="X", fund_name="X",
            fund_type="qof", oz_version="oz2", tract_id="17031840100",
            investment_type="real_estate", capital_gain_invested=1_000_000,
            investment_date="2027-01-01", fmv_at_investment=1_000_000,
            current_fmv=1_400_000, is_rural=False,
        ),
        current_fmv=1_400_000,
        exit_date=exit_dt,
    )
    print(f"{years:<12} {b.stepup_pct*100:.0f}%{'':<9} "
          f"${b.excluded_appreciation:<17,.0f} ${b.total_tax_benefit:,.0f}")


Hold (yrs)   Step-Up %    Exclusion ($)      Total Benefit ($)
------------------------------------------------------------
3            0%          $0                 $7,142
5            0%          $0                 $11,898
7            10%          $0                 $34,510
10           10%          $400,000           $129,710
15           10%          $400,000           $129,710
20           10%          $400,000           $129,710


## 9. Portfolio Tracking

In [11]:
portfolio = OZPortfolio(name="Family OZ Portfolio")

investments = [
    # P001 carries its OWN exit_date. That field has existed since 0.1.0 and
    # calculate_benefits() did not read it until now — a fully-specified
    # investment like this one was measured to today anyway and then REFUSED,
    # told it had omitted a field it had supplied. It is determinable from the
    # investment alone, with no argument at the call site.
    OZInvestment(
        id="P001", investor_name="Jay Patel",
        fund_name="Chicago South Side OZ Fund",
        fund_type="qof", oz_version="oz2",
        tract_id="17031840100", investment_type="real_estate",
        capital_gain_invested=750_000, investment_date="2027-03-15",
        fmv_at_investment=750_000, current_fmv=900_000,
        exit_date="2037-03-15",
        state="IL", is_rural=False,
    ),
    # P002 and P003 supply no exit date from either source. Their dates are
    # deliberately absurd — far beyond any plausible OZ horizon — so that they
    # stay not-yet-made no matter WHEN this notebook is run.
    #
    # They were 2027-06-01 and 2027-09-01, which made this whole section a
    # demonstration with an expiry date: after 2027-09-01 every investment
    # here would have been determinable and the cells below would have shown
    # nothing. tests/test_holding_period.py uses 2015 / 2099 for exactly this
    # reason.
    OZInvestment(
        id="P002", investor_name="Jay Patel",
        fund_name="Rural Illinois QORF",
        fund_type="qorf", oz_version="oz2",
        tract_id="17019000100", investment_type="real_estate",
        capital_gain_invested=500_000, investment_date="2099-06-01",
        fmv_at_investment=500_000, current_fmv=600_000,
        state="IL", is_rural=True,
    ),
    OZInvestment(
        id="P003", investor_name="Jay Patel",
        fund_name="Detroit OZ Business Fund",
        fund_type="qof", oz_version="oz2",
        tract_id="26163518300", investment_type="operating_business",
        capital_gain_invested=250_000, investment_date="2099-09-01",
        fmv_at_investment=250_000, current_fmv=300_000,
        state="MI", is_rural=False,
    ),
]

for inv in investments:
    portfolio.add(inv)

portfolio.summary()



OZ Investment Portfolio — Family OZ Portfolio

PORTFOLIO OVERVIEW
  Total Investments:     3
  Total Capital Gains:   $1.50MM
  OZ 1.0 Investments:    $0.00MM
  OZ 2.0 Investments:    $1.50MM
  Rural QORF:            $0.50MM

TAX BENEFITS (est. @ 23.8% cap gains rate)
  ⚠ PARTIAL — covers 1 of 3 investments ($0.75MM of $1.50MM in capital gains).
    This is NOT a portfolio total. See NOT DETERMINABLE below.
  Benefit (covered subset): $0.06MM
  % of covered gain:     8.2%

NOT DETERMINABLE (2 of 3)
  These investments are EXCLUDED from the figure above and are not zero —
  their benefit has no answer from the inputs given:
    • P002 (Rural Illinois QORF): $0.50MM
      Cannot calculate benefits for investment 'P002' ('Rural Illinois QORF'): no holding period exists. exit_date was not supplied — neither as an argument nor on the investment — so it defaulted to today (2026-08-02), which is BEFORE the investment date 2099-06-01: this investment has not been made yet. Pass exit_date=...,

### Why that report is PARTIAL

Three investments, two different situations, and the report keeps them apart.

**P001 is covered.** It carries its own `exit_date` (2037-03-15), so its
holding period is a fact available from the investment itself — no argument
needed. `calculate_benefits()` resolves the exit date as
**parameter → `investment.exit_date` → today**, and the middle leg is new:
until this release the field was never read, so P001 would have been measured
to *today*, gone negative, and been refused with "exit_date was not supplied"
— a statement about the caller's own input that was false. A portfolio of
fully-specified investments reported 100% NOT DETERMINABLE.

**P002 and P003 are excluded.** Neither supplies an exit date from either
source, and both are dated beyond today, so there is no holding period to
measure and therefore no benefit figure.

The package refuses to invent one. It does **not** clamp the holding period to
zero and report `$0.00`: a confident zero is indistinguishable from a genuine
zero-benefit result, which would be the same fabricated-certainty problem as
answering `False` for a tract nobody looked up.

Note what `summary()` does and does not do. It reports the covered subset
**explicitly scoped** ("covers 1 of 3"), computes the percentage against the
*covered* gain rather than the whole portfolio, names every excluded
investment, and never prints a bare "Total Tax Benefit" line.
`total_tax_benefits()` — which returns a bare `float` with nowhere to carry
that caveat — refuses outright:

In [12]:
from oztracker import OZCalculationError

try:
    portfolio.total_tax_benefits()
except OZCalculationError as e:
    print("total_tax_benefits() refused:\n")
    print(f"  {e}\n")

# Introspect coverage without catching anything:
print("Undeterminable members:")
for inv, reason in portfolio.undeterminable_benefits():
    print(f"  {inv.id}: {reason.split(':')[1].strip()}")

total_tax_benefits() refused:

  Cannot total tax benefits for portfolio 'Family OZ Portfolio': 2 of 3 investment(s) have no determinable holding period ('P002', 'P003'). Refusing to return a subtotal that would read as a complete total. First reason: Cannot calculate benefits for investment 'P002' ('Rural Illinois QORF'): no holding period exists. exit_date was not supplied — neither as an argument nor on the investment — so it defaulted to today (2026-08-02), which is BEFORE the investment date 2099-06-01: this investment has not been made yet. Pass exit_date=..., or set investment.exit_date, to model it.

Undeterminable members:
  P002: no holding period exists. exit_date was not supplied — neither as an argument nor on the investment — so it defaulted to today (2026-08-02), which is BEFORE the investment date 2099-06-01
  P003: no holding period exists. exit_date was not supplied — neither as an argument nor on the investment — so it defaulted to today (2026-08-02), which is BEFORE

### Supplying the missing exit dates makes the whole portfolio answerable

Give the two excluded investments an exit and the portfolio computes
completely. This is the intended workflow for not-yet-made investments — and
there are two ways to supply it, both shown below:

- set `investment.exit_date` on the object, as P001 already does — the exit
  that is *on record* for that investment;
- pass `exit_date=` to `calculate_benefits()` — the exit you are *modelling*
  right now.

When both are present the **parameter wins**, mirroring how `current_fmv`
already overrides `investment.current_fmv`: the per-call argument is the more
specific input. P001 demonstrates that in the first table — it holds
2037-03-15 on record and is modelled to 2039-03-15, and the 12.0y it reports
is the parameter's answer, not the field's.

> **A day-count caveat worth knowing.** `holding_years` is `days / 365.25`, so
> an exact calendar anniversary does not reliably clear the matching integer
> tier — a decade with only two leap days measures 9.9986 years and misses the
> 10-year exclusion. The sensitivity table in section 8 shows the same effect:
> its "5-year" row reports a 0% step-up because 2027-01-01 → 2032-01-01 is
> 4.9993 years. This is a rounding artifact of the calculator, not a statutory
> rule, and it is why the table below models 12 years rather than 10.

In [13]:
# Model every investment to a 12-year hold.
#
# 12 and not 10, deliberately. holding_years is measured as days / 365.25, so
# an exact 10-year anniversary is not reliably >= 10: a decade containing only
# two Feb-29s spans 3652 days = 9.9986 years and loses the 10-year exclusion.
# P002/P003 start in 2099 and their decade crosses 2100, a century year that
# is NOT a leap year — so at +10 they would print "10.0y" (rounded) with a $0
# exclusion, while P001's 2027-2037 decade cleared the bar. Two rows labelled
# identically, computed differently. Modelling to 12 years puts all three
# unambiguously past the tier.
HOLD_YEARS = 12

total = 0.0
print(f"{'ID':<6} {'Fund':<28} {'Hold':<7} {'Benefit':>11}  {'Exit date from'}")
print("-" * 82)
for inv in investments:
    exit_year = int(inv.investment_date[:4]) + HOLD_YEARS
    modelled_exit = f"{exit_year}{inv.investment_date[4:]}"
    b = calculate_benefits(inv, exit_date=modelled_exit)
    total += b.total_tax_benefit
    source = (f"parameter {modelled_exit} OVERRIDES field {inv.exit_date}"
              if inv.exit_date else f"parameter {modelled_exit} (no field set)")
    print(f"{inv.id:<6} {inv.fund_name:<28} {b.holding_years:>4.1f}y  "
          f"${b.total_tax_benefit:>10,.0f}  {source}")

print("-" * 82)
print(f"{'':<6} {'Total (all 3 determinable)':<28} {'':<7} ${total:>10,.0f}")
print("\nThis total is complete — every member had a determinable holding period.")

# The other route: set the field on the objects that lack one. The portfolio
# entry points read it through calculate_benefits()'s fallback, so nothing is
# undeterminable afterwards and total_tax_benefits() stops refusing.
for inv in investments:
    if inv.exit_date is None:
        inv.exit_date = (f"{int(inv.investment_date[:4]) + HOLD_YEARS}"
                         f"{inv.investment_date[4:]}")

print("\nAfter setting investment.exit_date on P002 and P003:")
print(f"  undeterminable members: {portfolio.undeterminable_benefits()}")
print(f"  total_tax_benefits():   ${portfolio.total_tax_benefits():,.0f}")

# P001 falls back to its own recorded 2037-03-15 here rather than the
# 2039-03-15 modelled above, since nothing overrides it now. The two totals
# still coincide, because every benefit tier this calculator models tops out
# at a 10-year hold — the extra two years change nothing. On a horizon that
# straddles a tier, they would differ.


ID     Fund                         Hold        Benefit  Exit date from
----------------------------------------------------------------------------------
P001   Chicago South Side OZ Fund   12.0y  $    61,582  parameter 2039-03-15 OVERRIDES field 2037-03-15
P002   Rural Illinois QORF          12.0y  $    63,665  parameter 2111-06-01 (no field set)
P003   Detroit OZ Business Fund     12.0y  $    20,528  parameter 2111-09-01 (no field set)
----------------------------------------------------------------------------------
       Total (all 3 determinable)           $   145,775

This total is complete — every member had a determinable holding period.

After setting investment.exit_date on P002 and P003:
  undeterminable members: []
  total_tax_benefits():   $145,775


In [14]:
df = portfolio.to_dataframe()
print("Portfolio DataFrame:")
print(df[["fund", "fund_type", "oz_version", "capital_gain",
          "is_rural", "state"]].to_string(index=False))


Portfolio DataFrame:
                      fund fund_type oz_version  capital_gain  is_rural state
Chicago South Side OZ Fund       qof        oz2        750000     False    IL
       Rural Illinois QORF      qorf        oz2        500000      True    IL
  Detroit OZ Business Fund       qof        oz2        250000     False    MI


## Summary

This notebook demonstrated the oz-tracker workflow that **actually works in
0.2.0**, and was explicit about the part that does not:

0. **Real lookup raises** — `OZ1Checker()` raises `OZDownloadError` rather than
   substituting sample data or answering `False`
1. **OZ 1.0 designation** — tri-state, on synthetic sample data
2. **OZ 2.0 eligibility** — tri-state, on synthetic sample data
3. **Tract comparison** — set arithmetic over the sample fixtures
4. **Eligibility rules** — pure threshold functions, fully functional
5-8. **QOF/QORF tax benefits, scenarios, sensitivity** — fully functional
9. **Portfolio tracking** — fully functional

**On the tract sections (1-3):** every number is synthetic. Do not quote them.
`NOT CONFIRMED` is the honest rendering of `None` and does not mean "not
designated" — it means the package could not answer. A `True` is a fact; the
absence of a `True` is not evidence of a negative.

**On the tax sections (5-9):** these compute from the inputs shown and perform
no lookup, so the outputs are genuine.

**Key context for investors:** the 2026 window matters — deploy under OZ 1.0
before 12/31/2026 or wait for OZ 2.0 from 1/1/2027. Rural QORF investments
carry a 30% basis step-up vs 10% for standard urban QOFs. (Modeled above from
statutory parameters; not tax advice.)

**GitHub:** https://github.com/Jaypatel1511/oz-tracker
**PyPI:** https://pypi.org/project/oz-tracker